In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, FloatType, LongType

spark = SparkSession.builder\
        .appName("e-commerce_bronze_transform")\
        .master("local[*]")\
        .config("spark.executor.memory", "2g")\
        .config("spark.driver.memory", "2g")\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/03 08:25:49 WARN Utils: Your hostname, Dune, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/03 08:25:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/03 08:25:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
data_path = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-enriched.csv"
output_dir = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/parquet/"

TODO List
1) Schema Enforcement (Evolution)
2) Data Quality Checks
    * Confirm email is in correct format \w@\w
    * Price is > 0
    * User Session is in \w{8}-\w{4}-\w{4}-\w{4}-\w{12} format
    * If category_code is not NULL, confirm the value is in ^\w+\.?(\w+\.)?(\w+)?$
    * Event Time is not Null

In [3]:
schema = StructType([
    StructField("event_time", DateType()),
    StructField("event_type", StringType()),
    StructField("product_id", IntegerType()),
    StructField("category_id", LongType()),
    StructField("category_code", StringType()),
    StructField("brand", StringType()),
    StructField("price", FloatType()),
    StructField("user_id", LongType()),
    StructField("user_session", StringType()),
    StructField("name", StringType()),
    StructField("username", StringType()),
    StructField("email", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("state_code", StringType()),
    StructField("zip_code", StringType()),
    StructField("sex", StringType()),
    StructField("product_name", StringType()),
    StructField("product_description", StringType())
])

In [4]:
df = spark.read.format("csv")\
    .schema(schema)\
    .option("header", "true")\
    .load(data_path)
print("Number of partitions:", df.rdd.getNumPartitions())

Number of partitions: 105


In [5]:
# Coalesce into fewer partitions
df.write.partitionBy("event_time").parquet(path=output_dir, mode="overwrite")

In [6]:
spark.stop()